# Step 11: Toy Models of Superposition

Neural networks have **fewer neurons than features**. A language model tracking thousands of
concepts (syntax, entities, facts, register, tone...) has a hidden layer of fixed width.
How does it manage?

The answer is **superposition**: features are stored as *nearly-orthogonal directions* in
activation space — not as individual neurons. A 512-neuron layer can represent thousands
of features, accepting small mutual interference, as long as features are sparse (rarely
active at the same time).

This explains **polysemantic neurons**: neurons that respond to multiple unrelated features.
They're not confused — they're doing efficient distributed coding.

This notebook reproduces the core results from:
> Elhage et al. (2022). *Toy Models of Superposition.* transformer-circuits.pub

We build the simplest possible model where superposition can be observed and studied.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from itertools import product

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

device = torch.device('cpu')  # CPU is fine for these tiny models

## The Setup

We have **n features** in the world. Each feature xᵢ ∈ [0, 1] is sparse:
it's nonzero with probability (1 - S), where S is the sparsity level.

We want to store these n features in a **bottleneck** of m dimensions (m < n).
Then read them back out.

```
x (n features)
  →  h = ReLU(Wx)       compress to m dims
  →  x̂ = W^T h + b      reconstruct n features
```

Loss = reconstruction error + sparsity pressure:
```
L = Σᵢ Iᵢ · (xᵢ - x̂ᵢ)²
```
where Iᵢ is the **importance** of feature i (some features matter more than others).

This is a one-hidden-layer autoencoder with tied weights and a ReLU bottleneck.
No bias on the encoder, one bias on the decoder.

The key questions:
- When does the model represent features as dedicated dimensions vs. in superposition?
- How does sparsity change this?
- What geometry do superposed features form?

In [ ]:
class ToyModel(nn.Module):
    """
    n features → m hidden dimensions → n features (reconstruction)
    Tied weights: encoder = W, decoder = W^T
    ReLU bottleneck enforces nonnegative hidden activations.
    """
    def __init__(self, n_features, n_hidden):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_hidden, n_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(n_features))
        self.n_features = n_features
        self.n_hidden = n_hidden

    def forward(self, x):
        # Encode: project to hidden space
        h = F.relu(x @ self.W.T)          # (batch, n_hidden)
        # Decode: project back with tied weights
        x_hat = h @ self.W + self.b       # (batch, n_features)
        return x_hat, h


def generate_sparse_data(n_samples, n_features, sparsity):
    """
    Generate sparse feature vectors.
    Each feature is nonzero with probability (1 - sparsity).
    Nonzero values are uniform in [0, 1].
    """
    # Mask: 1 if feature is active, 0 if not
    mask = torch.bernoulli(torch.full((n_samples, n_features), 1 - sparsity))
    values = torch.rand(n_samples, n_features)
    return mask * values


def train(model, n_features, sparsity, importance, n_steps=2000, batch_size=256, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    imp = torch.tensor(importance, dtype=torch.float32)  # (n_features,)

    for step in range(n_steps):
        x = generate_sparse_data(batch_size, n_features, sparsity)
        x_hat, _ = model(x)
        # Weighted reconstruction loss
        loss = (imp * (x - x_hat) ** 2).sum(dim=-1).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model


print("Model defined. n_features=5, n_hidden=2")
model = ToyModel(n_features=5, n_hidden=2)
print(f"W shape: {model.W.shape}  (n_hidden x n_features)")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

## Experiment 1: Dense Features → PCA (No Superposition)

When features are **dense** (frequently active), the cost of interference is high.
Two active features sharing a dimension will constantly interfere with each other.

In this regime the model does the sensible thing: **PCA**.
It finds the top m principal components and represents only those,
ignoring the rest. Each hidden dimension is dedicated to one direction in feature space.

We expect the weight vectors W to be nearly orthogonal.

In [ ]:
n_features = 5
n_hidden = 2
# All features equally important
importance = [1.0] * n_features

# --- Dense features (sparsity = 0.0 means always active) ---
model_dense = ToyModel(n_features, n_hidden)
train(model_dense, n_features, sparsity=0.0, importance=importance, n_steps=3000)

# --- Sparse features (sparsity = 0.9 means 90% of the time inactive) ---
model_sparse = ToyModel(n_features, n_hidden)
train(model_sparse, n_features, sparsity=0.9, importance=importance, n_steps=3000)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, title, sparsity in [
    (axes[0], model_dense,  'Dense features (S=0.0) → PCA', 0.0),
    (axes[1], model_sparse, 'Sparse features (S=0.9) → Superposition', 0.9),
]:
    W = model.W.detach().numpy()  # (2, 5)

    # Plot feature directions as arrows from origin in 2D hidden space
    colors = plt.cm.Set1(np.linspace(0, 0.8, n_features))
    for i in range(n_features):
        ax.arrow(0, 0, W[0, i], W[1, i],
                 head_width=0.03, head_length=0.02,
                 fc=colors[i], ec=colors[i], linewidth=2)
        ax.text(W[0, i] * 1.12, W[1, i] * 1.12, f'F{i}',
                color=colors[i], fontsize=10, ha='center', va='center')

    # Draw unit circle for reference
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), 'gray', alpha=0.3, linewidth=1)
    ax.axhline(0, color='gray', alpha=0.3); ax.axvline(0, color='gray', alpha=0.3)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Hidden dim 1'); ax.set_ylabel('Hidden dim 2')

    # Compute and display pairwise cosine similarities
    W_norm = W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-8)
    gram = W_norm.T @ W_norm
    off_diag = gram[np.triu_indices(n_features, k=1)]
    ax.set_xlabel(f'Hidden dim 1\nMean |cosine sim|={np.mean(np.abs(off_diag)):.3f}')

plt.suptitle('Feature Directions in 2D Hidden Space (5 features, 2 dims)', fontsize=13)
plt.tight_layout()
plt.show()
print("Saved: 11_dense_vs_sparse.png")

## What We Just Saw

**Dense features**: Two features claim the two available dimensions. The other three
features are essentially ignored (their weight vectors are near-zero). The represented
features are nearly orthogonal. This is PCA.

**Sparse features**: *All five features* are represented — even though there are only two
dimensions. They're arranged as nearly-equally-spaced directions (like a regular pentagon
inscribed on the unit circle). No feature is ignored, but each is stored at an angle to
the others — they interfere slightly. This is superposition.

The key intuition:
- When features are sparse, two features rarely activate *together*
- When only one feature fires, its direction is sufficient to reconstruct it even with superposition
- The rare interference (when two features fire simultaneously) is a small price to pay
  for representing more features

The ReLU is crucial: it can filter out small negative "phantom activations" that appear
as interference from non-active features.

## Experiment 2: The Phase Diagram

Superposition is a **phase transition** governed by sparsity and feature importance.
Let's map the phase diagram by varying both.

For each (sparsity, importance) pair, we train a model and measure whether each feature
is represented (|Wᵢ|² > threshold), and whether it's in superposition
(angle with nearest feature < threshold).

We'll use the simplest case: **n=2 features, m=1 dimension**.
In this case superposition means both features stored as opposite signs in one dimension.

In [ ]:
# Phase diagram: n=2 features, m=1 hidden dim
# Vary: sparsity S and importance of feature 2 (feature 1 always has importance 1.0)

def classify_2feat_1dim(sparsity, importance_ratio, n_steps=2000, n_seeds=3):
    """
    Returns the 'phase' for 2 features in 1 dimension:
      0 = neither feature represented
      1 = only feature 1 (dedicated)
      2 = only feature 2 (dedicated)
      3 = both in superposition (antipodal: W[0]>0, W[1]<0 or vice versa)
    """
    results = []
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        model = ToyModel(n_features=2, n_hidden=1)
        train(model, 2, sparsity=sparsity,
              importance=[1.0, importance_ratio], n_steps=n_steps)
        w = model.W.detach().numpy().flatten()  # shape (2,)
        norm = np.abs(w)
        threshold = 0.1
        f1 = norm[0] > threshold
        f2 = norm[1] > threshold
        opposite_sign = (w[0] * w[1] < 0)

        if f1 and f2 and opposite_sign:
            results.append(3)  # superposition
        elif f1 and f2 and not opposite_sign:
            results.append(1)  # ambiguous — count as feature 1
        elif f1:
            results.append(1)
        elif f2:
            results.append(2)
        else:
            results.append(0)
    # Return majority vote
    from collections import Counter
    return Counter(results).most_common(1)[0][0]


sparsity_vals = np.linspace(0.0, 0.99, 20)
importance_vals = np.linspace(0.1, 1.0, 15)

phase_grid = np.zeros((len(importance_vals), len(sparsity_vals)))
for i, imp in enumerate(importance_vals):
    for j, s in enumerate(sparsity_vals):
        phase_grid[i, j] = classify_2feat_1dim(s, imp)
    print(f"  importance_ratio={imp:.2f} done")

print("Phase grid computed.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

cmap = plt.cm.get_cmap('Set2', 4)
im = ax.imshow(phase_grid, origin='lower', aspect='auto', cmap=cmap,
               vmin=-0.5, vmax=3.5,
               extent=[sparsity_vals[0], sparsity_vals[-1],
                       importance_vals[0], importance_vals[-1]])

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.set_ticklabels(['Neither', 'F1 only (dedicated)', 'F2 only', 'Both (superposition)'])

ax.set_xlabel('Sparsity S (probability feature is inactive)', fontsize=12)
ax.set_ylabel('Importance of feature 2 (feature 1 = 1.0)', fontsize=12)
ax.set_title('Phase Diagram: 2 Features, 1 Hidden Dimension', fontsize=13)

ax.text(0.02, 0.95, 'Dense features\n→ only most important\n  feature represented',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
ax.text(0.65, 0.05, 'Sparse features\n→ both features in\n  superposition',
        transform=ax.transAxes, fontsize=9, va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()
print("Saved: 11_phase_diagram.png")

## What the Phase Diagram Shows

Three distinct regions:

1. **Low sparsity (left)**: Dense features → interference is costly → only the most important
   feature gets represented. The less important feature is ignored. This is the PCA regime.

2. **High sparsity, high importance (top right)**: Both features are important enough that
   even at moderate sparsity, the model represents them both in **superposition**:
   one feature stored as +W, the other as -W in the same dimension.
   They never fire together (sparsity ensures this), so they don't interfere.

3. **High sparsity, low importance (bottom right)**: Feature 2 is too unimportant to be
   worth representing even in superposition. Only feature 1 is learned.

The boundaries are **sharp** — this is a first-order phase transition, not a gradual shift.
A small change in sparsity can switch a feature from "not learned" to "in superposition"
discontinuously.

## Experiment 3: The Geometry of Superposition

When more features are packed into fewer dimensions, they don't arrange randomly.
They form specific **polytope geometries** — solutions to the Thomson problem
(how to arrange n electrons on a sphere to minimize repulsion):

| Features | Dimensions | Shape |
|---|---|---|
| 2 | 1 | Antipodal pair (±1) |
| 3 | 2 | Equilateral triangle |
| 4 | 3 | Tetrahedron |
| 5 | 2 | Regular pentagon |
| 6 | 2 | Regular hexagon |

Let's verify the triangle and pentagon cases directly.

In [ ]:
def train_and_get_directions(n_features, n_hidden, sparsity=0.85,
                              n_steps=4000, lr=5e-4, seed=0):
    torch.manual_seed(seed)
    importance = [1.0] * n_features
    model = ToyModel(n_features, n_hidden)
    train(model, n_features, sparsity, importance, n_steps=n_steps, lr=lr)
    W = model.W.detach().numpy()  # (n_hidden, n_features)
    # Normalize each feature direction
    norms = np.linalg.norm(W, axis=0, keepdims=True)
    W_norm = W / (norms + 1e-8)
    return W, W_norm, norms.flatten()


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
configs = [
    (3, 2, 'Triangle\n(3 features, 2 dims)'),
    (5, 2, 'Pentagon\n(5 features, 2 dims)'),
    (6, 2, 'Hexagon\n(6 features, 2 dims)'),
]

for ax, (n_feat, n_hid, title) in zip(axes, configs):
    W, W_norm, norms = train_and_get_directions(n_feat, n_hid)

    colors = plt.cm.Set1(np.linspace(0, 0.9, n_feat))

    # Plot unit circle
    theta = np.linspace(0, 2 * np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), 'lightgray', linewidth=1)

    # Plot feature directions
    for i in range(n_feat):
        scale = norms[i]
        ax.arrow(0, 0, W[0, i], W[1, i],
                 head_width=0.04, head_length=0.03,
                 fc=colors[i], ec=colors[i], linewidth=2, alpha=0.8)
        ax.text(W_norm[0, i] * 1.2, W_norm[1, i] * 1.2, f'F{i}',
                color=colors[i], fontsize=9, ha='center', va='center', fontweight='bold')

    ax.axhline(0, color='lightgray', linewidth=0.5)
    ax.axvline(0, color='lightgray', linewidth=0.5)
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=11)

    # Show pairwise cosine similarities
    gram = W_norm.T @ W_norm
    off = gram[np.triu_indices(n_feat, k=1)]
    ax.set_xlabel(f'Mean cosine sim = {np.mean(off):.3f}\n(ideal: {-1/(n_feat-1):.3f})')

plt.suptitle('Polytope Geometry of Superposition', fontsize=13)
plt.tight_layout()
plt.show()
print("Saved: 11_polytope_geometry.png")

## Reading the Geometry

The features arrange into **regular polytopes** — evenly-spaced directions on the circle
(or sphere in higher dimensions). This minimizes total pairwise interference, since the
average cosine similarity between equally-spaced directions on a sphere is -1/(n-1).

For n features in m dimensions, the **dimensions-per-feature ratio** D* = m/n settles
into discrete values corresponding to polytopes:
- Pentagon (5 in 2D): D* = 2/5 = 0.4
- Triangle (3 in 2D): D* = 2/3 ≈ 0.667
- Tetrahedron (4 in 3D): D* = 3/4 = 0.75

These D* values are "sticky" — as you sweep sparsity, the model jumps between polytope
configurations rather than smoothly interpolating. The geometry is determined by which
polytope minimizes the total interference cost at the given sparsity level.

## Experiment 4: Varying Feature Importance

Not all features are equally important. What happens when some features matter more?

Prediction: Important features get dedicated dimensions first. Less important features
pack into superposition. The least important features may be ignored entirely.

Let's test with 5 features, 2 dimensions, and decreasing importance: [1.0, 0.7, 0.4, 0.2, 0.1].

In [ ]:
n_features = 5
n_hidden = 2
importance_weights = [1.0, 0.7, 0.4, 0.2, 0.1]
sparsity = 0.9

torch.manual_seed(0)
model_imp = ToyModel(n_features, n_hidden)
train(model_imp, n_features, sparsity=sparsity, importance=importance_weights, n_steps=4000)

W = model_imp.W.detach().numpy()
norms = np.linalg.norm(W, axis=0)  # (n_features,)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: feature directions in 2D hidden space
ax = axes[0]
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'lightgray', linewidth=1)
colors = plt.cm.RdYlGn(np.linspace(0.8, 0.2, n_features))  # green=important, red=less
for i in range(n_features):
    ax.arrow(0, 0, W[0, i], W[1, i],
             head_width=0.04, head_length=0.03,
             fc=colors[i], ec=colors[i], linewidth=2, alpha=0.9)
    ax.text(W[0, i] * 1.2, W[1, i] * 1.2,
            f'F{i}\n(I={importance_weights[i]})',
            color=colors[i], fontsize=8, ha='center', va='center', fontweight='bold')
ax.axhline(0, color='lightgray', linewidth=0.5)
ax.axvline(0, color='lightgray', linewidth=0.5)
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
ax.set_aspect('equal')
ax.set_title('Feature Directions (green=high importance)', fontsize=11)

# Right: feature norms (how strongly each feature is represented)
ax = axes[1]
bar_colors = colors
bars = ax.bar(range(n_features), norms, color=bar_colors)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Full representation')
ax.set_xlabel('Feature index')
ax.set_ylabel('||Wᵢ|| (representation strength)')
ax.set_title('How Strongly Each Feature Is Represented', fontsize=11)
ax.set_xticks(range(n_features))
ax.set_xticklabels([f'F{i}\nI={importance_weights[i]}' for i in range(n_features)])
ax.legend()
for i, (bar, norm) in enumerate(zip(bars, norms)):
    ax.text(bar.get_x() + bar.get_width()/2, norm + 0.01, f'{norm:.2f}',
            ha='center', va='bottom', fontsize=9)

plt.suptitle('Varying Feature Importance: 5 Features, 2 Dims, Sparsity=0.9', fontsize=12)
plt.tight_layout()
plt.show()

## Experiment 5: Computation in Superposition

Superposition isn't just for storage — models can also *compute* in superposition.

Here we test whether a model can compute **absolute value** on each feature,
even when features are stored in superposition. The function f: x → |x|
would normally require separate neurons for positive and negative inputs.

With superposition, a 2-layer model can approximate abs() for many features
simultaneously using what the paper calls the **asymmetric superposition motif**.

In [ ]:
class AbsModel(nn.Module):
    """
    Learns to compute absolute value: x → |x|
    Input: n features in [-1, 1] (can be negative)
    Hidden: m neurons (ReLU)
    Output: n values (should approximate |xᵢ|)
    """
    def __init__(self, n_features, n_hidden):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(n_hidden, n_features) * 0.3)
        self.b1 = nn.Parameter(torch.zeros(n_hidden))
        self.W2 = nn.Parameter(torch.randn(n_features, n_hidden) * 0.3)
        self.b2 = nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        h = F.relu(x @ self.W1.T + self.b1)
        out = h @ self.W2.T + self.b2
        return out


def train_abs(model, n_features, sparsity, n_steps=3000, batch_size=512, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(n_steps):
        # Sparse inputs in [-1, 1]
        mask = torch.bernoulli(torch.full((batch_size, n_features), 1 - sparsity))
        values = torch.rand(batch_size, n_features) * 2 - 1
        x = mask * values
        target = torch.abs(x)
        pred = model(x)
        loss = F.mse_loss(pred, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model


# Compare models with different hidden size ratios
n_features = 10
sparsity = 0.8
configs = [
    (10, 'n_hidden=n_features\n(one neuron per feature)'),
    (5,  'n_hidden=n_features/2\n(superposition needed)'),
    (2,  'n_hidden=2\n(heavy superposition)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (n_hidden, label) in zip(axes, configs):
    torch.manual_seed(42)
    model = AbsModel(n_features, n_hidden)
    train_abs(model, n_features, sparsity)

    # Evaluate on 1000 test points
    with torch.no_grad():
        mask = torch.bernoulli(torch.full((1000, n_features), 1 - sparsity))
        x_test = mask * (torch.rand(1000, n_features) * 2 - 1)
        pred = model(x_test)
        target = torch.abs(x_test)
        mse = F.mse_loss(pred, target).item()

    # Scatter plot: predicted vs. true |x| for feature 0
    nonzero_mask = (x_test[:, 0].abs() > 0.05).numpy()
    true_vals = target[:, 0].numpy()[nonzero_mask]
    pred_vals = pred[:, 0].numpy()[nonzero_mask]

    ax.scatter(true_vals, pred_vals, alpha=0.3, s=10, color='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', alpha=0.7, label='Perfect')
    ax.set_xlabel('True |x₀|')
    ax.set_ylabel('Predicted |x₀|')
    ax.set_title(label, fontsize=10)
    ax.text(0.05, 0.92, f'MSE={mse:.4f}', transform=ax.transAxes, fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle(f'Computing abs() in Superposition ({n_features} features, sparsity={sparsity})',
             fontsize=12)
plt.tight_layout()
plt.show()
print("Note: even the heavy-superposition model approximates abs() reasonably well.")

## Experiment 6: Polysemantic Neurons

Superposition directly causes **polysemantic neurons** — neurons that respond to multiple
unrelated features. Let's visualize which features activate each neuron in a superposed model.

In [ ]:
n_features = 8
n_hidden = 3  # Heavy superposition: 8 features in 3 neurons
sparsity = 0.9

torch.manual_seed(7)
model_poly = ToyModel(n_features, n_hidden)
train(model_poly, n_features, sparsity=sparsity,
      importance=[1.0] * n_features, n_steps=4000)

# For each neuron, show which features it responds to most strongly
# W shape: (n_hidden, n_features) — W[neuron, feature]
W = model_poly.W.detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: heatmap of W
ax = axes[0]
im = ax.imshow(np.abs(W), cmap='Blues', aspect='auto')
ax.set_xlabel('Feature index')
ax.set_ylabel('Neuron index')
ax.set_xticks(range(n_features))
ax.set_yticks(range(n_hidden))
ax.set_xticklabels([f'F{i}' for i in range(n_features)])
ax.set_yticklabels([f'N{i}' for i in range(n_hidden)])
plt.colorbar(im, ax=ax, label='|Weight|')
ax.set_title(f'|W|: Neuron↔Feature Weights\n({n_hidden} neurons, {n_features} features)', fontsize=11)

# Annotate cells with actual weight values
for i in range(n_hidden):
    for j in range(n_features):
        ax.text(j, i, f'{W[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if abs(W[i,j]) > 0.5 else 'black')

# Right: for each neuron, rank features by response strength
ax = axes[1]
colors = plt.cm.Set2(np.linspace(0, 1, n_hidden))
width = 0.25
x = np.arange(n_features)
for i in range(n_hidden):
    ax.bar(x + i * width, np.abs(W[i]), width,
           label=f'Neuron {i}', color=colors[i], alpha=0.8)

ax.set_xlabel('Feature index')
ax.set_ylabel('|Weight| (neuron response strength)')
ax.set_title('Polysemanticity: Each Neuron Responds\nto Multiple Features', fontsize=11)
ax.set_xticks(x + width)
ax.set_xticklabels([f'F{i}' for i in range(n_features)])
ax.legend()
ax.axhline(0.3, color='gray', linestyle='--', alpha=0.5, label='threshold')

plt.suptitle(f'Polysemantic Neurons: {n_features} Features in {n_hidden} Neurons (Sparsity={sparsity})',
             fontsize=12)
plt.tight_layout()
plt.show()
print()
print("Each neuron with multiple strong weights is POLYSEMANTIC.")
print("This is not a bug — it's superposition doing efficient coding.")

## The Safety Implication: Enumerative Safety

Polysemantic neurons make **interpretability hard**. When neuron 2 responds to features
F0, F3, and F6, you cannot understand what the model is doing by looking at neurons.
You need to find the underlying features — but those don't correspond to neurons.

From the paper's safety analysis:

> *"Enumerative safety" would require the ability to enumerate all features learned by a model
> — a universal quantifier over the model's computational units. This would allow verifying
> that no dangerous circuit exists anywhere. Superposition blocks feature enumeration:
> more features than neurons means features ≠ neurons.*

**Three paths forward** (explored in the next notebook):
1. Train models that don't use superposition (L1 regularization, MoE architectures)
2. Apply sparse autoencoders post-hoc to *find* the features in superposition
3. Hybrid: reduce superposition architecturally, then decompose residuals

The next notebook implements option 2.

## Key Takeaways

| Concept | What we learned |
|---|---|
| **Superposition** | Neural networks represent more features than neurons via nearly-orthogonal directions |
| **Dense features → PCA** | When features are frequently active, only top-m features get dedicated dimensions |
| **Sparse features → superposition** | Rare co-occurrence makes interference cheap; the ReLU filters it out |
| **Phase transition** | The shift between PCA and superposition is sharp, not gradual |
| **Polytope geometry** | Superposed features arrange as regular polytopes (triangles, pentagons, tetrahedra) |
| **Polysemantic neurons** | A direct consequence: each neuron codes for multiple features |
| **Computation in superposition** | Models can compute functions (like abs()) even with superposed features |
| **Safety implication** | Polysemanticity blocks feature enumeration — the prerequisite for safety auditing |

**Next**: Notebook 12 shows how sparse autoencoders can *undo* superposition by finding the
underlying monosemantic features.